# Day 1 — Machine Learning & MLOps in Practice

In this module, we implement a classical machine learning pipeline for customer churn prediction and evaluate it. We cover data cleaning, feature engineering, model training, and serializing models for deployment.

## Objectives
- Build a baseline classification model predicting telecom customer churn.
- Establish a clean feature preprocessing pipeline.
- Train and compare Logistic Regression, Decision Trees, and Random Forests.
- **Extended Implementation:** Hyperparameter optimization using GridSearchCV, Feature Importance analysis, Confusion Matrix & ROC-AUC curves, and a custom interactive client predictor.

In [ ]:
# Step 1: Install correct library versions
!pip install pandas==2.2.2 scikit-learn==1.5.1 joblib==1.4.2 matplotlib==3.9.0 seaborn==0.13.2 -q
print("✅ Libraries installed.")

## Dataset Ingestion
We download the Telco Customer Churn dataset, which contains ~7000 customer records and a target classification variable `Churn` (Yes/No).

In [ ]:
# Step 2: Download the dataset
!wget -q https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv -O telco_churn.csv
print("✅ Dataset downloaded.")

import pandas as pd
df = pd.read_csv("telco_churn.csv")
print("Shape:", df.shape)
df.head(3)

## Data Cleaning and Preprocessing
We check data types and notice that `TotalCharges` is read as a string object because of spaces. We convert it to numeric and handle missing values.

In [ ]:
import numpy as np

# Convert space to NaN, cast to float, and impute using median
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Drop ID column as it contains no pattern details
df = df.drop(columns=['customerID'])

# Encode categorical variables using LabelEncoder
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=['object']).columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("✅ Preprocessing complete. Encoded columns:", list(categorical_cols))

## Train-Test Split and Model Training
We split the dataset into a training set (80%) and testing set (20%), then train three classification algorithms to establish baseline scores.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train algorithms
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"{name} Accuracy: {accuracy_score(y_test, preds):.4f}")

## Model Serialization (MLOps)
We save our trained model and label encoders using `joblib` for deployment and persistent classification use.

In [ ]:
import joblib

best_model = models["Random Forest"]
joblib.dump(best_model, 'churn_rf_model.pkl')
joblib.dump(label_encoders, 'label_encoders.pkl')
print("✅ Model and encoders saved successfully.")

---
# Extended Implementation
Below are implementation techniques covering parameter tuning, feature importances, detailed metrics, and inference interfaces.

### 1. Hyperparameter Tuning with GridSearchCV
We optimize parameters to find the best Random Forest setup.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)
print(f"Tuned Random Forest Test Accuracy: {accuracy_score(y_test, grid_search.best_estimator_.predict(X_test)):.4f}")

### 2. Feature Importance Visualization
We analyze feature importance values to see which columns contribute most to predicting customer churn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

best_rf = grid_search.best_estimator_
importances = best_rf.feature_importances_
feature_names = X.columns

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_df, palette='viridis')
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### 3. Detailed Metrics: Confusion Matrix & ROC-AUC Curve

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

# Confusion Matrix
y_preds = best_rf.predict(X_test)
cm = confusion_matrix(y_test, y_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# ROC Curve
y_probs = best_rf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
auc = roc_auc_score(y_test, y_probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

### 4. Local Interactive Customer Churn Predictor
We create a helper function that takes custom JSON data, preprocesses it, and runs the loaded model to output predictions.

In [ ]:
def predict_custom_customer(customer_data, model_path='churn_rf_model.pkl', encoder_path='label_encoders.pkl'):
    import joblib
    import pandas as pd
    
    # Load assets
    model = joblib.load(model_path)
    encoders = joblib.load(encoder_path)
    
    # Convert input to DataFrame
    input_df = pd.DataFrame([customer_data])
    
    # Preprocess
    for col in input_df.columns:
        if col in encoders:
            try:
                # If category exists in label encoder, transform it
                input_df[col] = encoders[col].transform(input_df[col])
            except ValueError:
                # If unseen category, handle gracefully by assigning default -1
                input_df[col] = -1
                
    # Ensure correct columns order matches the training features
    feature_order = X.columns
    input_df = input_df[feature_order]
    
    # Predict
    prob = model.predict_proba(input_df)[0][1]
    churn_pred = model.predict(input_df)[0]
    
    return {
        "Churn Prediction": "Yes" if churn_pred == 1 else "No",
        "Churn Probability": f"{prob * 100:.2f}%"
    }

# Mock user data payload matching features
mock_customer = {
    'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'Yes', 'Dependents': 'No',
    'tenure': 2, 'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'DSL',
    'OnlineSecurity': 'Yes', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No',
    'StreamingTV': 'No', 'StreamingMovies': 'No', 'Contract': 'Month-to-month', 'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check', 'MonthlyCharges': 70.85, 'TotalCharges': 140.7
}

prediction = predict_custom_customer(mock_customer)
print("Interactive Prediction Result for Mock Customer:")
print(prediction)